# Self-play benchmark — parallel vs vectorized (Option B)

**Group 501** | Colman College | DL Final Project

Decides whether to flip a 9x9 run to `self_play_mode: "vectorized"`.

Run the cells top to bottom. Section 1 mirrors the training notebooks: it
installs `requirements.txt` into this kernel (the container's `/usr/bin/python3`
ships without torch) and then reports what torch will actually run on.

**Requires an idle GPU** — section 2 checks. Run this next to a live training job
and both engines are measured under contention, which invalidates the result.

---
## 1. Environment Setup
Run once per kernel session.

In [ ]:
# 1.1 — Locate repo and install dependencies

import os, sys

# Find repo root (works from any starting directory)
REPO_DIR = None
for candidate in [
    os.getcwd(),                               # already in repo root
    os.path.join(os.getcwd(), "dl-quoridor"),  # Jupyter workspace root
    os.path.dirname(os.getcwd()),              # running from notebooks/
]:
    if os.path.exists(os.path.join(candidate, ".git")):
        REPO_DIR = candidate
        break

assert REPO_DIR is not None, (
    "Could not find dl-quoridor repo. "
    "Make sure the notebook is inside the repo or one level above it."
)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo: {REPO_DIR}")
print(f"Python: {sys.executable}")

# Deliberately NOT running `git checkout . && git pull` like the training
# notebooks do: this benchmark is meant to be run from a feature branch, and
# `git checkout .` would discard your working changes. Confirm the branch:
!git -C {REPO_DIR} log --oneline -1
!git -C {REPO_DIR} status -sb | head -1

# Install requirements (--ignore-installed handles system-managed packages)
!pip install -r requirements.txt -q --ignore-installed
print("Dependencies installed.")

In [ ]:
# 1.2 — Detect hardware; confirm torch sees the GPU

from scripts.bench_self_play import describe_device, run_bench, DEFAULTS

DEVICE = describe_device("auto")

If 1.2 printed `cuda_available=False`, **stop here**. Both engines would be
CPU-bound, the vectorized driver would run unpipelined, and the numbers would
say nothing about how this box behaves during real training.

---
## 2. Is the GPU free?

Any python process listed below means something is already training. Benchmarking
under contention penalises the worker-pool path most, which biases the result
toward vectorized.

In [ ]:
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

---
## 3. Smoke test (~1 min)

Tiny run, just to confirm both engines execute in this kernel. The ratio here is
meaningless — 5x5, 25 sims, no warmup convergence.

In [ ]:
_ = run_bench(games=4, sims=25, num_workers=4, max_moves=40, warmup_games=1)

---
## 4. The real measurement

Matches the 9x9 N=4 training config. Takes a while — 50 games at 800 sims, three
times per engine, plus warmup.

Lower `repeat` to 1 for a faster (noisier) answer.

In [ ]:
BENCH = dict(
    board_size=9, num_players=4, walls=10,
    num_channels=128, num_res_blocks=8,
    sims=800, games=50, vec_games=64,
    num_workers=32, batch_size=256,
    max_moves=300, explore_moves=15,
    device="auto", warmup_games=2, repeat=3,
)
results = run_bench(**BENCH)
results

---
## 5. Confirm the ordering effect is not the story

If step 4's margin is inside ~10%, re-run with the engines swapped. A result that
flips with order is not a real difference.

In [ ]:
results_swapped = run_bench(**{**BENCH, "order": "vectorized,parallel"})

for label, r in [("original", results), ("swapped", results_swapped)]:
    if {"parallel", "vectorized"} <= set(r):
        print(f"{label:9s}: parallel/vectorized = "
              f"{r['parallel'] / r['vectorized']:.2f}x")

---
## 6. Verdict

Flip the run to `"vectorized"` only if **both** orderings agree that vectorized is
faster, by a margin you would notice.

To switch:

1. Stop the run.
2. `configs/config_9x9.json` → `"self_play_mode": "vectorized"`, `"vec_games": 64`.
3. Resume — weights persist via `latest.pt`; only the replay buffer refills.

The games played after the switch differ from what the parallel engine would have
produced (different exploration-noise stream), but their quality does not: the
vectorized driver runs exact sequential MCTS with no virtual-loss approximation.